# Latent diffusion model based on High-Resolution Image Synthesis with Latent Diffusion Models

## Things to do
 - **Try cross attention in THE UNET**
 - understand **perceptual loss**
 


There is 2 stages in our DMs although this wasn't formalized and thought while making the models in DM models (DDPM a. &):
- perceptual compression (1st stage)
- semantic compression (2 stage)

DMs require a lot of compute. Two distinct issues are at play. First, as likelihood-based models, DMs are mode-covering and end up spending an excessive share of their parametric capacity on modeling imperceptible high-frequency details. Connecting this to the DDPM timestep structure: during the forward process, high-frequency content is destroyed first (since white Gaussian noise dominates the low-power high-frequency components quickly), while the semantic/low-frequency structure persists much longer. Equivalently, in the reverse direction: at large t the model reconstructs coarse semantic content, and at small t it refines fine perceptual details. The reweighted simplified objective of Ho et al. partially addresses this by down-weighting the loss at small t, where the model would otherwise overfit on perceptually irrelevant content. Second, even with this reweighting, every U-Net evaluation is expensive because it operates on full-resolution RGB tensors, and both training and sampling require many such evaluations. LDM addresses both issues by factoring the problem: a perceptual autoencoder handles the perceptual compression stage upstream, and the diffusion model runs in a compressed latent space where the remaining bits are semantically meaningful.

Why do we need to be sure that the variance in the latent space is low?
Remember that in our usual DDPM/DDIM **We normalize the data** (goes to [-1,1]) so that we correctly destroy the images after some steps with a N(0,I) noise. Nethertheless, our latent space doesn't have this constraint yet and our latent images could be in [-1528,200] (higher variance in the data and not centered) where our noise schedule would be then incompatible. 

## Let's first do the decoder/encoder


In [1]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
import torch.nn.functional as F
import torch.nn as nn


SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),                    # convert image to HxWxC tensors with floats in [0,1]
    transforms.Normalize((0.5,), (0.5,)),     # [-1, 1], cause we do (img - 0.5)/0.5 to get a centered around 0 and unit distribution
])

train_set = datasets.MNIST(
    root="./data",        
    train=True,
    download=True,
    transform=transform,
)

test_set = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform,
)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=0)
test_loader  = DataLoader(test_set,  batch_size=128, shuffle=False, num_workers=0)


100%|██████████| 9.91M/9.91M [00:01<00:00, 5.49MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 127kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.24MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.51MB/s]


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device") 

Using cuda device


In [ ]:
class RMSnorm(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.scale = dim**0.5
        self.g = nn.Parameter(torch.ones(1, dim, 1, 1))
    def forward(self,x):
        return F.normalize(x, dim=1) * self.g * self.scale #Pytorch formula normalize is the L2 and then we have the

class Resblock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim=None):
        super().__init__()

        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm1 = RMSnorm(out_ch) #we use RMSnorm which is more efficient
        self.act1 = nn.SiLU()
        
        if time_dim is not None:
            self.time_proj = nn.Sequential(nn.SiLU(),nn.Linear(time_dim, 2 * out_ch)) 
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.norm2 = RMSnorm(out_ch)
        self.act2 = nn.SiLU()
        self.res_conv = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity() #we need output and skip(x) to have the same dim
    def forward(self, x, t_emb=None):
        #block 1
        if t_emb is not None:
            t_context = self.time_proj(t_emb) #(B, 2 * out_ch)
            t_context = t_context[:,:,None,None] # We need the time_context to match output (B, 2 * out_ch, 1, 1)
            scale, shift = t_context.chunk(2, dim=1) #tuple (B, out_ch,1,1)
        
        h1 = self.conv1(x)
        h1 = self.norm1(h1)
        if t_emb is not None:
            h1 = h1 * (1 + scale) + shift 
        h1 = self.act1(h1)
        #block 2
        h2 = self.conv2(h1)
        h2 = self.norm2(h2)
        h2 = self.act2(h2)
        return (h2 + self.res_conv(x)) /np.sqrt(2) #adding variance normalisation with 1/sqrt(2)  

def Downsample(dima, dimb):
    return nn.Conv2d(dima, dimb, kernel_size=2, stride=2) # we divide by 2 the image feature resolution
def Upsample(dima, dimb):
    return nn.Sequential(
        nn.Upsample(scale_factor=2, mode="bilinear"), #got helped by google on that one
        nn.Conv2d(dima, dimb, 3, padding=1),
    )


In [5]:
######  I HAVE TO ADD CROSS ATTENTION 

class Encoder(nn.Module):
    def __init__(self, dim, latent_ch,start_ch=1, dim_mults=[1,2], beta=1e-6):
        # voir si dim est pas obvious?
        super().__init__()
        self.beta = beta
        # comprendre pourquoi les ModuleList 
        dims = [dim] + [dim * m for m in dim_mults]
        in_out = list(zip(dims[:-1], dims[1:])) #giving couple of (in_ch, out_ch)
        self.init_conv = nn.Conv2d(start_ch, dim, kernel_size=1)
        # DownBlocks
        self.downs = nn.ModuleList([])
        for (in_ch, out_ch) in in_out:
            down = nn.ModuleList([Resblock(in_ch, in_ch), Resblock(in_ch, in_ch), Downsample(in_ch, out_ch)])
            self.downs.append(down)

        #MidBlocks
        mid_dim = dims[-1]
        self.mid_block1 = Resblock(mid_dim, mid_dim)
        #self.mid_block2 = Resblock(mid_dim, mid_dim) # 4 mid_blocks might be overkill

        # latent projection (might be easier for the regularization)
        self.latent_proj = nn.Conv2d(mid_dim, 2 * latent_ch, kernel_size=1) # we could determine dim with latent_dim and thus
                                                                            # having 1 parameter, but this gives more flexibility
        #BE CAREFUL WITH THE 2*LATENT CH, ITS FOR 1 TYPE OF REGULARIZATION                                 
    def forward(self, x):
        """
        x; (B, C, H, W) -> x_out; (B, [2 *] latent_ch, ?, ?)
        """
        x = self.init_conv(x)
        for down in self.downs:
            res1 , res2, downsam = down
            x = res1(x)
            x = res2(x)
            x = downsam(x)
        x = self.mid_block1(x)
        #x = self.mid_block2(x)
        x = self.latent_proj(x) 
        return x
    


In [16]:
def encode_loss(mu, log_var, beta=1e-6):
        var = torch.exp(log_var)
        loss = (0.5*(mu**2 + var - log_var - 1)).mean()
        return beta * loss

In [7]:
# #test
# x = torch.randn( (5, 1, 28, 28))
# test_mod = encoder(64, 4, dim_mults=[1,2])
# x = test_mod(x)
# x.shape

### Decoder's turn

In [8]:
class Decoder(nn.Module):
    def __init__(self, dim, latent_ch,start_ch=1, dim_mults=[1,2]):
        super().__init__()
        dims = [dim] + [dim * m for m in dim_mults]
        in_out = list(zip(dims[:-1], dims[1:])) #giving couple of (in_ch, out_ch)
        
        self.init_proj = nn.Conv2d(latent_ch, dims[-1], kernel_size=1) #we need to project the latent space to the right dimension for the mid_block
        
        #MidBlocks
        mid_dim = dims[-1]
        self.mid_block1 = Resblock(mid_dim, mid_dim)
        #self.mid_block2 = Resblock(mid_dim, mid_dim)

        # UpBlocks
        self.ups = nn.ModuleList([])
        for (out_ch, in_ch) in reversed(in_out): #careful with the order in_ch, out_ch
            up = nn.ModuleList([Upsample(in_ch, out_ch), Resblock(out_ch, out_ch), Resblock(out_ch, out_ch)]) 
            self.ups.append(up)

        self.final_conv = nn.Conv2d(dim, start_ch, kernel_size=1) #we need to project back to the right number of channels

    def forward(self,x):
        """
        x; (B, latent_ch, ?, ?) -> x_out; (B, C, H, W)
        """
        x = self.init_proj(x)
        x = self.mid_block1(x)
        #x = self.mid_block2(x)
        for up in self.ups:
            upsam , res1, res2 = up
            x = upsam(x)
            x = res1(x)
            x = res2(x)
        x = self.final_conv(x)
        return torch.tanh(x) #our image must be in [-1,1], It's not the same as in DDPM, because we were predicting eps ~ N(0,I) which isn't bounded
        


In [9]:
# # testdecoder
# x = torch.randn((5,8,7,7))
# print(x.mean(), x.std())
# test_dec = decoder(64, 4)
# x= test_dec(x)
# print(x.mean(), x.std(), x.shape)

# Strange thing

At first, when I saw the decoder, I thought it was strange that we sample from a normal distribution for each image? Each vector in the latent space should be a sort of gaussian distribution? Looks bad at first sight, but that's because we're not seeing it right. 
The encoder doesn't give z, it gives z | x and the goal of our diffusion model is to capture p(z) = E_x(z | x) where we modelize z|x as a gaussian just like in the DDPM. Now the point is that every distribution can be approximate by a mixture of gaussians which assure us that we'll learn some complex distribution. 

# Perceptual loss

Perceptual loss looks interesting, but the easy method to implement it won't work well on MNIST since it's small images. Let's do some perceptual loss later from scratch, but we'll keep it simple here with only a pixel loss on our images.


## Adversarial training

In [10]:
class PatchDiscriminator(nn.Module):
    def __init__(self, in_channels=1): # might be a problem due to the dimensions? 
        super().__init__()
        
        self.net = nn.Sequential(   # soucis entre ModuleList et nn.Sequential? 
            # (B, 3, H, W)
            nn.Conv2d(in_channels, 64, 4, stride=2, padding=1),
            nn.GroupNorm(8, 64),
            nn.SiLU(),
            # (B, 64, H/2, W/2)
            nn.Conv2d(64, 128, 4, stride=2, padding=1),
            nn.GroupNorm(8, 128),
            nn.SiLU(),            
            # (B, 128, H/4, W/4)
            nn.Conv2d(128, 1, 4, padding=1),
        )
    
    def forward(self, x):
        return self.net(x)  # pas de sigmoid, on utilise BCEWithLogitsLoss
    
    def adv_loss(self, gen, x):
        fake, real = self.forward(gen), self.forward(x) #careful, we want to train the discriminator here, not the gen, so gen.detach()
        return F.relu(1 - real).mean() + F.relu(1 + fake).mean()
    
    


In [11]:
# x = torch.randn((5,1,28,28))
# test_mod = PatchDiscriminator(1)
# x = test_mod(x)
# x.shape

## Note

This kind of adversarial and multiple loss looks really harder to train. You have to be careful about a lot of things, it's easily unstable and 

In [19]:

encoder = torch.compile(Encoder(dim=64, latent_ch=4, start_ch=1, dim_mults=[1,2])).to(device) 
decoder = torch.compile(Decoder(dim=64, latent_ch=4)).to(device)
discriminator = torch.compile(PatchDiscriminator(1)).to(device)

models = encoder, decoder, discriminator

from torch.utils.data import Subset, DataLoader
tiny_set = Subset(train_set, range(5000)) #using the whole dataset is too long to train
tiny_loader = DataLoader(tiny_set, batch_size=128, shuffle=True)

optimizer1 = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=5e-4)
optimizer2 = torch.optim.Adam(discriminator.parameters(), lr=5e-4)

optimizers = optimizer1, optimizer2

losses = []  
def train_loops(dataloader, models, optimizers, use_disc=False):
    opt1, opt2 = optimizers
    enc, dec, disc_mod = models
    scaler = torch.amp.GradScaler('cuda')
    for _, (X, _) in enumerate(dataloader):
        X = X.to(device)
        with torch.autocast("cuda"):             
            adv_loss, g_loss = 0, 0
            z = enc(X)
            mu, log_var = z.chunk(2, dim=1)
            eps = torch.randn_like(mu, device=mu.device)
            if use_disc:
                disc_mod.requires_grad_(True)
                gen = dec(mu + eps * torch.exp(0.5 * log_var)).detach()
                adv_loss = disc_mod.adv_loss(gen, X)
                scaler.scale(adv_loss).backward()
                scaler.step(opt2)
                opt2.zero_grad()

            disc_mod.requires_grad_(False)
            n_gen = dec(mu + eps * torch.exp(0.5 * log_var))
            if use_disc:
                g_loss = -torch.mean(disc_mod(n_gen))
            kl_loss = encode_loss(mu, log_var, beta=1e-3)
            loss = kl_loss + F.mse_loss(n_gen, X) + 0.1 * g_loss

        scaler.scale(loss).backward()
        scaler.step(opt1)
        scaler.update()
        opt1.zero_grad()
        losses.append(loss.item())


epochs = 100
wup_ep = 4
for epoch in range(epochs):
    use_disc= False if epoch<= wup_ep else True

    train_loops(tiny_loader, models, optimizers, use_disc)
    if epoch % 10 == 0:
        print(f"epoch {epoch}: loss = {losses[-1]:.6f}")
plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel("iteration")
plt.ylabel("loss")
plt.yscale("log")  
plt.title("Training loss")
plt.grid(True, alpha=0.3)
plt.show()



W0516 08:00:34.765000 5746 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


RuntimeError: Function CompiledFunctionBackward returned an invalid gradient at index 5 - got [64] but expected shape compatible with [1, 64, 1, 1]

In [15]:
losses

[0.6290736794471741,
 0.6253185272216797,
 0.4082181751728058,
 0.4230613112449646,
 0.4189107120037079,
 0.39078429341316223,
 0.7832607626914978,
 0.7862687706947327,
 0.37150710821151733,
 0.41793787479400635,
 0.42759087681770325,
 0.4206591844558716,
 0.42342543601989746,
 0.4478219449520111,
 0.40985724329948425,
 0.4157685935497284,
 0.4177098870277405,
 0.42684975266456604,
 0.4149217903614044,
 0.38166242837905884,
 0.3893602192401886,
 0.3623935282230377,
 0.3743137717247009,
 0.3672139644622803,
 0.3708643913269043,
 0.36843377351760864,
 0.3739182949066162,
 0.35648855566978455,
 0.39626267552375793,
 0.3732317388057709,
 0.35353291034698486,
 0.3556833863258362,
 0.3624204397201538,
 0.34165680408477783,
 0.3471992313861847,
 0.3331548571586609,
 0.30447104573249817,
 0.2891359329223633,
 0.28706085681915283,
 0.2730962336063385,
 0.24364261329174042,
 0.2558635473251343,
 0.26186782121658325,
 0.2448311746120453,
 0.21892942488193512,
 0.2375900000333786,
 0.2126122862100